# 2D Likelihood Contours: CW Source Parameters vs Pulsar Distance

This notebook generates **N** random pulsars, injects **M** continuous
gravitational-wave (CGW) sources, and sweeps the PTA log-likelihood over a 2D
grid of *(one CW-source parameter of source 0)* × *(the distance of one target
pulsar)*. Pick which parameter(s) to sweep in the `SWEEPS` list below:

| mode | swept parameter | notes |
|---|---|---|
| `"strain"` | $\log_{10} h$ (`cw0_log10_h`) | broad, non-resonant |
| `"frequency"` | $\log_{10} f_{\rm gw}$ (`cw0_log10_fgw`) | narrow window — off-resonance the likelihood collapses |
| `"sky_ra"` | source RA $\phi_{\rm gw}$ (`cw0_gwphi`) | ~1 mrad window resolves the pulsar-term fringes |
| `"sky_dec"` | source DEC in degrees (via `cw0_cos_gwtheta` $= \sin$ DEC) | ~1 mrad window |
| `"cgw_distance"` | luminosity distance $D_L$ (Mpc) | mapped to strain via `log10_strain_from_binary` at fixed chirp mass: $h_0 \propto 1/D_L$ |

**Convention:** `PX` is parallax in mas (PINT / `types.py` convention). The
sweep iterates over `PX` in mas; the y-axis is displayed in kpc via
`d = 1 / PX_mas`. Inside `CWInjector` the same conversion is applied so the
Ellis+2012 pulsar-term phase operates on physical distance.

This notebook consolidates the former `likelihood_contour_{strain,frequency,
skypos,cgw_distance}_vs_distance` notebooks — the PTA setup was identical in
all four; only the swept parameter differed.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt

In [ ]:
from __future__ import annotations

from loguru import logger

from io import StringIO

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import jaxpint.par as jpar

from jaxpint.pta.likelihood import pta_logL
from jaxpint.pta.signals.cw import log10_strain_from_binary
from jaxpint.notebook_utils import (
    build_cw_injectors,
    generate_random_par,
    inject_and_build_config,
    plot_2d_delta_logL,
    setup_synthetic_pta,
    sweep_1d_logL,
    sweep_2d_logL,
)

# ---- Configuration ----
N_PULSARS = 10
M_CW_SOURCES = 1
N_TOAS = 200
START_MJD = 57000.0
END_MJD = 60000.0        # ~8 yr span (the sky sweeps sharpen with a longer one)
TOA_ERROR = 1e-8         # 10 ns
FREQ = 1400.0            # MHz
SEED = 42

# Which parameter(s) of CW source 0 to sweep against the target pulsar's
# distance. Any subset of: "strain", "frequency", "sky_ra", "sky_dec",
# "cgw_distance". Each mode is one 2D sweep of GRID_DENSITY^2 likelihood
# evaluations — trim the list or lower GRID_DENSITY on a CPU.
SWEEPS = ["strain", "frequency", "sky_ra", "sky_dec", "cgw_distance"]
GRID_DENSITY = 2000

# CGW chirp mass used by the "cgw_distance" mode to map D_L -> strain
# (a 10^9 Msun SMBH binary puts the injected source at a few tens of Mpc).
LOG10_MC = 9.0

## Generate random pulsars

Each pulsar gets a random sky position, spin frequency, spindown rate, and
distance. The par file includes spindown (F0, F1), astrometry (RAJ, DECJ, PX),
and unscaled white noise (EFAC = 1).

In [ ]:
rng = np.random.default_rng(SEED)

par_strings = [generate_random_par(idx, rng, start_mjd=START_MJD) for idx in range(N_PULSARS)]
models = [jpar.get_model(StringIO(p)) for p in par_strings]

print(f"Generated {N_PULSARS} pulsars")
print(f"Example .par:\n{par_strings[0]}")

## Generate fake TOAs and convert to JaxPINT

In [ ]:
synthetic = setup_synthetic_pta(
    models,
    start_mjd=START_MJD, end_mjd=END_MJD,
    n_toas=N_TOAS, toa_error_s=TOA_ERROR, freq_mhz=FREQ,
)
pp_tuple = synthetic.pulsar_params_list

for i, model in enumerate(models):
    px_mas = float(pp_tuple[i].param_value("PX"))
    distance_kpc = 1.0 / px_mas
    f0 = float(pp_tuple[i].param_value("F0"))
    print(
        f"  Pulsar {i}: {model.metadata["PSR"]:>20s}  "
        f"PX={px_mas:.3f} mas (d={distance_kpc:.2f} kpc)  F0={f0:.1f} Hz"
    )

print(f"\nAll {N_PULSARS} pulsars loaded.")

## Set up M CW sources and inject into TOAs

We place M continuous gravitational-wave sources at random sky locations with
random nHz-band GW frequencies and strain amplitude $h = 10^{-14}$, then
record the true injected parameters of source 0 (the swept source).

In [ ]:
cw_injectors, _ = build_cw_injectors(
    models, n_sources=M_CW_SOURCES, rng=rng, log10_h=-14.0,
)

for m, inj in enumerate(cw_injectors):
    print(
        f"  CW source {m}: cos_gwtheta={inj.param_spec['cos_gwtheta']:.3f}, "
        f"gwphi={inj.param_spec['gwphi']:.3f}, "
        f"log10_fgw={inj.param_spec['log10_fgw']:.2f}"
    )

gp, config = inject_and_build_config(synthetic, cw_injectors)

# True injected parameters of source 0.
TRUE_LOG10_H = float(cw_injectors[0].param_spec["log10_h"])
TRUE_LOG10_FGW = float(cw_injectors[0].param_spec["log10_fgw"])
TRUE_GWPHI = float(cw_injectors[0].param_spec["gwphi"])
TRUE_COS_GWTHETA = float(cw_injectors[0].param_spec["cos_gwtheta"])
TRUE_DEC_DEG = float(np.degrees(np.arcsin(TRUE_COS_GWTHETA)))

# "cgw_distance" mode: the injected strain, chirp mass, and frequency fix the
# true luminosity distance. h0 ∝ 1/D_L, so the strain at 1 Mpc sets the offset:
# log10_h = log10_h(1 Mpc) - log10(D_L / Mpc).
log10_h_at_1mpc = float(log10_strain_from_binary(LOG10_MC, 0.0, TRUE_LOG10_FGW))
TRUE_LOG10_DL = log10_h_at_1mpc - TRUE_LOG10_H

print(f"\nPTA config built with {M_CW_SOURCES} CW sources.")
print(f"Global params: {gp.names}")
print(
    f"True source 0: log10_h={TRUE_LOG10_H:.2f}, log10_fgw={TRUE_LOG10_FGW:.4f}, "
    f"RA={TRUE_GWPHI:.4f} rad, DEC={TRUE_DEC_DEG:.3f} deg, "
    f"D_L={10**TRUE_LOG10_DL:.1f} Mpc (M_c=10^{LOG10_MC:.1f} Msun)"
)

## Sweep specifications

Every mode shares the same y-axis: the target pulsar's distance, swept over a
narrow window (the pulsar-term phase oscillates rapidly with distance). The
distance grid is built in kpc for the plot axis and inverted to mas for the
likelihood, so rows are flipped at plot time to present kpc ascending.

Each spec maps a display-space grid `grid` onto the underlying global
parameter `param` through `to_param` (identity except: DEC → `cos_gwtheta =
sin(DEC)`; $D_L$ → strain via `log10_strain_from_binary`). `x_display`
converts grid values to plot units (only `"cgw_distance"` uses it, to show
Mpc on a log axis). `grid_1d`, where present, adds a wide 1D slice through
the true distance.

In [ ]:
TARGET_PULSAR = 0
true_px_mas = float(pp_tuple[TARGET_PULSAR].param_value("PX"))
true_distance = 1.0 / true_px_mas  # kpc
print(f"Pulsar {TARGET_PULSAR} true distance: {true_distance:.3f} kpc "
      f"(PX = {true_px_mas:.4f} mas)")

half_window_px = 0.005  # kpc (y-axis window in distance)
distance_grid = np.linspace(
    true_distance - half_window_px,
    true_distance + half_window_px,
    GRID_DENSITY,
)
px_mas_grid = 1.0 / distance_grid

_identity = lambda x: x

SWEEP_SPECS = {
    "strain": dict(
        param="cw0_log10_h",
        grid=np.linspace(TRUE_LOG10_H - 0.72, TRUE_LOG10_H + 0.2, GRID_DENSITY),
        to_param=_identity,
        true_x=TRUE_LOG10_H,
        xlabel=r"$\log_{10}(h)$",
        xscale="linear",
        title="GW strain vs pulsar distance",
        grid_1d=None,
        x_display=_identity,
    ),
    "frequency": dict(
        param="cw0_log10_fgw",
        grid=np.linspace(TRUE_LOG10_FGW - 0.05, TRUE_LOG10_FGW + 0.05, GRID_DENSITY),
        to_param=_identity,
        true_x=TRUE_LOG10_FGW,
        xlabel=r"$\log_{10}(f_{\rm gw}/\rm Hz)$",
        xscale="linear",
        title="GW frequency vs pulsar distance",
        grid_1d=np.linspace(TRUE_LOG10_FGW - 2.0, TRUE_LOG10_FGW + 2.0, 200_000),
        x_display=_identity,
    ),
    "sky_ra": dict(
        param="cw0_gwphi",
        grid=np.linspace(TRUE_GWPHI - 1e-2, TRUE_GWPHI + 1e-2, GRID_DENSITY),
        to_param=_identity,
        true_x=TRUE_GWPHI,
        xlabel=r"CW source RA $\phi_{\rm gw}$ (rad)",
        xscale="linear",
        title="CW RA vs pulsar distance",
        grid_1d=None,
        x_display=_identity,
    ),
    "sky_dec": dict(
        param="cw0_cos_gwtheta",
        grid=np.linspace(
            TRUE_DEC_DEG - np.degrees(1e-2), TRUE_DEC_DEG + np.degrees(1e-2), GRID_DENSITY
        ),
        to_param=lambda dec_deg: jnp.sin(jnp.radians(dec_deg)),
        true_x=TRUE_DEC_DEG,
        xlabel="CW source DEC (deg)",
        xscale="linear",
        title="CW DEC vs pulsar distance",
        grid_1d=None,
        x_display=_identity,
    ),
    "cgw_distance": dict(
        param="cw0_log10_h",
        grid=np.linspace(TRUE_LOG10_DL - 0.5, TRUE_LOG10_DL + 0.5, 200),
        to_param=lambda ld: log10_strain_from_binary(LOG10_MC, ld, TRUE_LOG10_FGW),
        true_x=TRUE_LOG10_DL,
        xlabel=r"CGW luminosity distance $D_L$ (Mpc)",
        xscale="log",
        title="CGW distance vs pulsar distance",
        grid_1d=np.linspace(TRUE_LOG10_DL - 1.0, TRUE_LOG10_DL + 1.0, 2000),
        x_display=lambda g: 10.0 ** np.asarray(g),
    ),
}


def make_eval(spec):
    """Build eval_logL(x, px_mas) for one sweep spec (vmap/JIT friendly)."""
    param, to_param = spec["param"], spec["to_param"]

    def eval_logL(x, px_mas_val):
        gp_mod = gp.with_value(param, to_param(x))
        pp_mod_0 = pp_tuple[TARGET_PULSAR].with_value("PX", px_mas_val)
        pp_mod = pp_tuple[:TARGET_PULSAR] + (pp_mod_0,) + pp_tuple[TARGET_PULSAR + 1:]
        return pta_logL(gp_mod, pp_mod, config)

    return eval_logL

## Run the 2D sweeps

One colormesh per entry in `SWEEPS`. The red star marks the injected truth.

In [ ]:
logL_2d_results = {}

for name in SWEEPS:
    spec = SWEEP_SPECS[name]
    grid = spec["grid"]
    print(f"[{name}] computing {len(grid)} x {len(px_mas_grid)} = "
          f"{len(grid) * len(px_mas_grid)} likelihood evaluations...")
    logL_2d = sweep_2d_logL(make_eval(spec), grid, px_mas_grid)
    logL_2d_results[name] = logL_2d

    # Rows follow px_mas_grid (descending in kpc); flip to present kpc ascending.
    fig, ax = plt.subplots(figsize=(10, 7))
    mesh = plot_2d_delta_logL(
        ax,
        spec["x_display"](grid),
        distance_grid[::-1],
        logL_2d[::-1, :],
        true_xy=(spec["x_display"](spec["true_x"]), true_distance),
    )
    ax.set_xscale(spec["xscale"])
    ax.set_xlabel(spec["xlabel"], fontsize=13)
    ax.set_ylabel("Pulsar distance (kpc)", fontsize=13)
    ax.set_title(
        f"PTA likelihood: {spec['title']}\n"
        f"({N_PULSARS} pulsars, {M_CW_SOURCES} CW sources, "
        f"sweeping source 0 & pulsar {TARGET_PULSAR} distance)",
        fontsize=13,
    )
    ax.legend(fontsize=12, loc="upper left")
    ax.tick_params(labelsize=11)
    fig.colorbar(mesh, ax=ax, label=r"$\Delta$ log-likelihood")
    fig.tight_layout()
plt.show()

## 1D slices at the true pulsar distance

For the modes that define a wide `grid_1d` (frequency and CGW distance),
slice through the 2D surface at the true distance to show how the likelihood
depends on the swept parameter alone.

In [ ]:
for name in SWEEPS:
    spec = SWEEP_SPECS[name]
    if spec["grid_1d"] is None:
        continue
    grid_1d = spec["grid_1d"]
    eval_2d = make_eval(spec)

    def _eval_1d(x, _px_mas=jnp.float64(true_px_mas), _eval=eval_2d):
        return _eval(x, _px_mas)

    print(f"[{name}] 1D slice: {len(grid_1d)} evaluations...")
    logL_1d = sweep_1d_logL(_eval_1d, grid_1d)
    delta_1d = logL_1d - logL_1d.max()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(spec["x_display"](grid_1d), delta_1d, linewidth=1.5)
    ax.axvline(
        spec["x_display"](spec["true_x"]), color="black", linestyle="--",
        linewidth=1.5, label="True value",
    )
    ax.set_xscale(spec["xscale"])
    ax.set_xlabel(spec["xlabel"], fontsize=13)
    ax.set_ylabel(r"$\Delta$ Log-likelihood", fontsize=13)
    ax.set_title(
        f"{spec['title']} — 1D slice at true pulsar distance "
        f"(d = {true_distance:.3f} kpc)",
        fontsize=14,
    )
    ax.legend(fontsize=11)
    ax.tick_params(labelsize=11)
    fig.tight_layout()
plt.show()